# Retrieval Demo

- **Nội dung chính:** Báo cáo cơ sở dữ liệu và demo truy suất đoạn theo điều khoản
- **Phạm vi:** Truy vấn, lọc theo điều/chương, kiểm tra dữ liệu đã index trong ChromaDB

> **Yêu cầu:** Đã chạy xong `Ingestion_pipeline_test.ipynb`.


In [16]:
# === Cell 1: Setup self-contained — không phụ thuộc src/ ===
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

EMBED_MODEL = "huyydangg/DEk21_hcmute_embedding"
COLLECTION_NAME = "legal_chunks"


def load_embedder(env_path: Path | None = None) -> SentenceTransformer:
    """Load SentenceTransformer embedding model."""
    if env_path:
        load_dotenv(env_path)
    hf_token = os.getenv("HF_TOKEN")
    return SentenceTransformer(EMBED_MODEL, token=hf_token)


def get_collection(chroma_dir: Path, collection_name: str = COLLECTION_NAME):
    """Get or create ChromaDB collection."""
    client = chromadb.PersistentClient(path=str(chroma_dir))
    return client.get_or_create_collection(collection_name)


def search(
    query: str,
    chroma_dir: Path,
    embedder: SentenceTransformer,
    top_k: int = 10,
    where: dict | None = None,
    collection_name: str = COLLECTION_NAME,
):
    """Vector search in ChromaDB and return normalized result list."""
    query_tokenized = ViTokenizer.tokenize(query.lower())
    query_vector = embedder.encode([query_tokenized], convert_to_numpy=True)[0]

    collection = get_collection(chroma_dir, collection_name)
    results = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=top_k,
        where=where,
        include=["metadatas", "documents", "distances"],
    )

    rows = []
    if results.get("ids") and len(results["ids"]) > 0:
        for i, chunk_id in enumerate(results["ids"][0]):
            distance = results["distances"][0][i] if results.get("distances") else 0.0
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": results["documents"][0][i],
                    "metadata": results["metadatas"][0][i] if results.get("metadatas") else {},
                    "distance": distance,
                    "score": 1 - distance,
                }
            )
    return rows


def get_by_filter(
    chroma_dir: Path,
    where: dict | None = None,
    limit: int = 200,
    collection_name: str = COLLECTION_NAME,
):
    """Metadata filter lookup in ChromaDB."""
    collection = get_collection(chroma_dir, collection_name)
    results = collection.get(where=where, limit=limit, include=["metadatas", "documents"])

    rows = []
    if results.get("ids"):
        for i, chunk_id in enumerate(results["ids"]):
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": results["documents"][i] if i < len(results.get("documents", [])) else "",
                    "metadata": results["metadatas"][i] if i < len(results.get("metadatas", [])) else {},
                }
            )
    return rows


CHROMA_DIR = ROOT / "chroma_db"
assert CHROMA_DIR.exists(), "Chưa có ChromaDB — chạy Ingestion_pipeline_test.ipynb trước!"

collection = get_collection(CHROMA_DIR)
total = collection.count()
print(f"[OK] ChromaDB: {total} chunks trong collection 'legal_chunks'")

embedder = load_embedder(ROOT / ".env")
print(f"[OK] Embedding model: {embedder.get_sentence_embedding_dimension()} dim")

[OK] ChromaDB: 593 chunks trong collection 'legal_chunks'


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 788.53it/s, Materializing param=pooler.dense.weight]                               


[OK] Embedding model: 768 dim


---

## Phần 1 — Báo cáo Cơ sở dữ liệu (Index Report)


In [17]:
# === Cell 2: Thống kê tổng quan ChromaDB (không dùng chunk_type) ===

all_data = collection.get(include=["metadatas", "documents"])
all_meta = all_data["metadatas"]
all_docs = all_data["documents"]

total_chunks = len(all_meta)
doc_counts  = Counter(m["doc_id"] for m in all_meta)
char_lens   = [len(d) for d in all_docs]
chuong_counts = Counter(m.get("chuong_so", "0") for m in all_meta)

SEP = "=" * 65
print(SEP)
print("  BAO CAO CO SO DU LIEU - ChromaDB")
print(SEP)
print(f"  Tong chunks : {total_chunks}")
print(f"  So van ban  : {len(doc_counts)}")
print()

print("-- Chunks theo van ban --")
for doc, cnt in sorted(doc_counts.items()):
    print(f"  {doc:<40} {cnt:>5}")
print()

print("-- Chunks theo chuong_so --")
for chuong, cnt in sorted(chuong_counts.items(), key=lambda x: x[0]):
    label = f"Chuong {chuong}" if chuong != "0" else "ngoai chuong"
    print(f"  {label:<20} {cnt:>5}")
print()

if char_lens:
    print("-- Do dai chunk --")
    print(f"  Min: {min(char_lens)} | Max: {max(char_lens)} | Avg: {sum(char_lens)//len(char_lens)} ky tu")
print(SEP)

  BAO CAO CO SO DU LIEU - ChromaDB
  Tong chunks : 593
  So van ban  : 10

-- Chunks theo van ban --
  112_2025_QH15_586814                        58
  114_2025_QH15_658530                        46
  116_2025_QH15_666020                        45
  127_2025_QH15_686325                       180
  133_2025_QH15_675211                        27
  135_2025_QH15_675213                        95
  143_2025_QH15_681550                        52
  148_2025_QH15_675262                        48
  sample v1                                   21
  sample v2                                   21

-- Chunks theo chuong_so --
  ngoai chuong            42
  Chuong I                90
  Chuong II              128
  Chuong III              49
  Chuong IV               91
  Chuong IX                9
  Chuong V                67
  Chuong VI               40
  Chuong VII              23
  Chuong VIII             12
  Chuong X                 9
  Chuong XI                3
  Chuong XII               5
  C

In [18]:
# === Cell 3: Bản đồ cấu trúc — số Điều mỗi Chương ===

doc_chuong = defaultdict(lambda: defaultdict(int))
for m in all_meta:
    doc_chuong[m["doc_id"]][m.get("chuong_so", "0")] += 1

print("BAN DO CAU TRUC: So Dieu moi Chuong\n")
for doc_id, chuong_map in sorted(doc_chuong.items()):
    total_dieu = sum(chuong_map.values())
    print(f"Doc {doc_id}  ({total_dieu} dieu)")
    for chuong_so, cnt in sorted(chuong_map.items()):
        label = f"Chuong {chuong_so}" if chuong_so != "0" else "(ngoai chuong)"
        print(f"   {label:<15}  {cnt:>3} dieu")
    print()

BAN DO CAU TRUC: So Dieu moi Chuong

Doc 112_2025_QH15_586814  (58 dieu)
   Chuong I          16 dieu
   Chuong II         15 dieu
   Chuong III         9 dieu
   Chuong IV          8 dieu
   Chuong V           6 dieu
   Chuong VI          4 dieu

Doc 114_2025_QH15_658530  (46 dieu)
   Chuong I          14 dieu
   Chuong II         12 dieu
   Chuong III         7 dieu
   Chuong IV          4 dieu
   Chuong V           6 dieu
   Chuong VI          3 dieu

Doc 116_2025_QH15_666020  (45 dieu)
   Chuong I           7 dieu
   Chuong II          5 dieu
   Chuong III        10 dieu
   Chuong IV          4 dieu
   Chuong V           3 dieu
   Chuong VI          9 dieu
   Chuong VII         4 dieu
   Chuong VIII        3 dieu

Doc 127_2025_QH15_686325  (180 dieu)
   Chuong I          15 dieu
   Chuong II         56 dieu
   Chuong III         7 dieu
   Chuong IV         25 dieu
   Chuong IX          9 dieu
   Chuong V          12 dieu
   Chuong VI          7 dieu
   Chuong VII         4 dieu
   

---

## Phần 2 — Demo Truy suất Theo Điều Khoản


In [19]:
# === Cell 4: Hàm hiển thị kết quả ===

def show_result(rank, r):
    """In thong tin mot ket qua truy van."""
    meta = r["metadata"]
    chuong = f"Chuong {meta.get('chuong_so', '0')}" if meta.get("chuong_so", "0") != "0" else "(ngoai chuong)"
    score_str = f"  |  Score: {r['score']:.4f}" if "score" in r else ""

    print("-" * 65)
    print(f"  [{rank}] Dieu {meta.get('article_number', '')} - {meta.get('article_title', '(khong co tieu de)')}")
    print(f"       Score :{score_str or ' N/A'}")
    print(f"       Nguon : {meta.get('doc_id', '')}  |  {chuong}")
    preview = r["text"][:300].strip()
    for line in preview.split("\n")[:6]:
        print(f"         {line}")
    if len(r["text"]) > 300:
        print(f"         ... [{len(r['text'])} ky tu]")

print("[OK] show_result() san sang")

[OK] show_result() san sang


In [20]:
# === Cell 5: Demo 1 — Semantic Search ===
# ✏️ Thay câu hỏi ở đây:
QUERY = "điều kiện để được cấp phép xây dựng"
TOP_K = 5

results = search(QUERY, CHROMA_DIR, embedder=embedder, top_k=TOP_K)

print(f'🔍 Câu hỏi: "{QUERY}"  →  Top {TOP_K}\n')
for i, r in enumerate(results, 1):
    show_result(i, r)
print("─" * 65)

🔍 Câu hỏi: "điều kiện để được cấp phép xây dựng"  →  Top 5

-----------------------------------------------------------------
  [1] Dieu 43 - Quy định chung về cấp giấy phép xây dựng
       Score :  |  Score: -23.4286
       Nguon : 135_2025_QH15_675213  |  Chuong III
         Điều 43. Quy định chung về cấp giấy phép xây dựng
         1. Giấy phép xây dựng bao gồm các loại sau đây:
         a) Giấy phép xây dựng mới;
         b) Giấy phép sửa chữa, cải tạo, di dời công trình;
         c) Giấy phép xây dựng có thời hạn.
         2. Trước khi khởi công xây dựng công trình, chủ đầu tư phải có giấy phép xây dựng trừ các
         ... [3929 ky tu]
-----------------------------------------------------------------
  [2] Dieu 44 - Cấp giấy phép xây dựng
       Score :  |  Score: -23.8157
       Nguon : 135_2025_QH15_675213  |  Chuong III
         Điều 44. Cấp giấy phép xây dựng
         1. Điều kiện để cấp giấy phép xây dựng đối với các loại giấy phép quy định tại khoản 1 Điều 43 của Luật này, 

In [21]:
# === Cell 6: Demo 1b — Nhiều câu hỏi mẫu ===

SAMPLE_QUERIES = [
    "quyền và nghĩa vụ của người sử dụng đất",
    "xử phạt vi phạm hành chính trong lĩnh vực môi trường",
    "điều kiện thành lập doanh nghiệp",
    "trách nhiệm của cơ quan nhà nước",
]

print("🔍 Top 1 cho mỗi câu hỏi mẫu\n")
for q in SAMPLE_QUERIES:
    res = search(q, CHROMA_DIR, embedder=embedder, top_k=1)
    if res:
        r = res[0]
        m = r["metadata"]
        print(f"  ❓ {q}")
        print(f"     → dieu_{m.get('article_number','')} | {m.get('article_title','')[:50]} | score={r['score']:.3f} | {m.get('doc_id','')}")
        print()

🔍 Top 1 cho mỗi câu hỏi mẫu



  ❓ quyền và nghĩa vụ của người sử dụng đất
     → dieu_10 | Quyền Và Nghĩa Vụ Của Các Bên | score=-45.734 | sample v2

  ❓ xử phạt vi phạm hành chính trong lĩnh vực môi trường
     → dieu_114 | Xử lý người chấp hành án phạt quản chế vi phạm ngh | score=-52.139 | 127_2025_QH15_686325

  ❓ điều kiện thành lập doanh nghiệp
     → dieu_19 | Đầu tư thành lập tổ chức kinh tế | score=-38.480 | 143_2025_QH15_681550

  ❓ trách nhiệm của cơ quan nhà nước
     → dieu_32 | Cung cấp dịch vụ công trực tuyến | score=-39.339 | 148_2025_QH15_675262



In [22]:
# === Cell 7: Demo 2 — Filter Lookup theo Điều ===
# ✏️ Thay đổi:
FILTER_ARTICLE = "5"
FILTER_DOC     = None  # None = tất cả văn bản

where = {"article_number": FILTER_ARTICLE}
if FILTER_DOC:
    where = {"$and": [{"article_number": FILTER_ARTICLE}, {"doc_id": FILTER_DOC}]}

results = get_by_filter(CHROMA_DIR, where=where)

print(f"🔎 Điều {FILTER_ARTICLE} | Văn bản: {FILTER_DOC or '(tất cả)'} → {len(results)} kết quả\n")
for i, r in enumerate(results, 1):
    show_result(i, r)
if not results:
    print("  ⚠️ Không tìm thấy.")
print("─" * 65)

🔎 Điều 5 | Văn bản: (tất cả) → 10 kết quả

-----------------------------------------------------------------
  [1] Dieu 5 - Hệ thống quy hoạch
       Score : N/A
       Nguon : 112_2025_QH15_586814  |  Chuong I
         Điều 5. Hệ thống quy hoạch
         1. Hệ thống quy hoạch bao gồm:
         a) Quy hoạch cấp quốc gia, gồm: quy hoạch tổng thể quốc gia, quy hoạch không gian biển quốc gia, quy hoạch sử dụng đất quốc gia, quy hoạch ngành;
         b) Quy hoạch vùng. Chính phủ xác định các vùng cần lập quy hoạch;
         c) Quy hoạch tỉnh;
         d) Quy hoạc
         ... [3731 ky tu]
-----------------------------------------------------------------
  [2] Dieu 5 - Quản lý nhà nước về phòng bệnh
       Score : N/A
       Nguon : 114_2025_QH15_658530  |  Chuong I
         Điều 5. Quản lý nhà nước về phòng bệnh
         1. Nội dung quản lý nhà nước về phòng bệnh bao gồm:
         a) Xây dựng và chỉ đạo thực hiện chiến lược, quy hoạch, kế hoạch, chương trình, đề án, dự án về phòng bệnh;
  

In [23]:
# === Cell 8: Demo 2b — Lọc tất cả Điều trong một Chương ===
# Thay doi:
FILTER_CHUONG = "I"
FILTER_DOC    = None

where = {"chuong_so": FILTER_CHUONG}
if FILTER_DOC:
    where = {"$and": [{"chuong_so": FILTER_CHUONG}, {"doc_id": FILTER_DOC}]}

results = get_by_filter(CHROMA_DIR, where=where)

print(f"Chuong {FILTER_CHUONG} | {FILTER_DOC or '(tat ca)'} -> {len(results)} dieu\n")
print(f"  {'STT':>4}  {'Dieu':>8}  {'Tieu de':<45}  {'Van ban'}")
print(f"  {'-'*4}  {'-'*8}  {'-'*45}  {'-'*30}")
for i, r in enumerate(results, 1):
    m = r["metadata"]
    print(f"  {i:>4}  {m.get('article_number',''):>8}  {(m.get('article_title','-'))[:45]:<45}  {m.get('doc_id','')[:30]}")

if results:
    print("\n--- Chi tiet dieu dau tien ---")
    show_result(1, results[0])
print("-" * 65)

Chuong I | (tat ca) -> 90 dieu

   STT      Dieu  Tieu de                                        Van ban
  ----  --------  ---------------------------------------------  ------------------------------
     1         1  Phạm vi điều chỉnh                             112_2025_QH15_586814
     2         2  Đối tượng áp dụng                              112_2025_QH15_586814
     3         3  Giải thích từ ngữ                              112_2025_QH15_586814
     4         4  Nguyên tắc cơ bản trong hoạt động quy hoạch    112_2025_QH15_586814
     5         5  Hệ thống quy hoạch                             112_2025_QH15_586814
     6         6  Nguyên tắc xác định quy hoạch phải điều chỉnh  112_2025_QH15_586814
     7         7  Thời kỳ, tầm nhìn quy hoạch                    112_2025_QH15_586814
     8         8  Sơ đồ quy hoạch, bản đồ quy hoạch              112_2025_QH15_586814
     9         9  Trình tự lập, thẩm định, quyết định hoặc phê   112_2025_QH15_586814
    10        10  Chi phí

In [24]:
# === Cell 9: Demo 3 — Xem toàn bộ nội dung một Điều ===
# Thay doi:
VIEW_ARTICLE = "1"

results = get_by_filter(CHROMA_DIR, where={"article_number": VIEW_ARTICLE})

if not results:
    print(f"⚠️ Khong tim thay Dieu {VIEW_ARTICLE}")
else:
    r = results[0]
    m = r["metadata"]
    chuong = f"Chuong {m.get('chuong_so','0')}" if m.get("chuong_so", "0") != "0" else "(ngoai chuong)"
    print(f"{'='*65}")
    print(f"  Dieu {m.get('article_number','')} - {m.get('article_title', '(khong co)')}")
    print(f"  Nguon: {m.get('doc_id','')}  |  {chuong}")
    print(f"{'-'*65}")
    print(r["text"])
    print(f"{'='*65}")
    if len(results) > 1:
        print(f"\n  Co {len(results)-1} van ban khac co Dieu {VIEW_ARTICLE}:")
        for r2 in results[1:]:
            print(f"     - {r2['metadata'].get('doc_id','')}: {r2['metadata'].get('article_title','')}")

  Dieu 1 - Phạm vi điều chỉnh
  Nguon: 112_2025_QH15_586814  |  Chuong I
-----------------------------------------------------------------
Điều 1. Phạm vi điều chỉnh
Luật này quy định hệ thống quy hoạch; việc lập, thẩm định, quyết định hoặc phê duyệt, công bố, cung cấp thông tin, thực hiện, đánh giá và điều chỉnh quy hoạch; quản lý nhà nước về hoạt động quy hoạch.

  Co 9 van ban khac co Dieu 1:
     - 114_2025_QH15_658530: Phạm vi điều chỉnh, đối tượng áp dụng
     - 116_2025_QH15_666020: Phạm vi điều chỉnh và đối tượng áp dụng
     - 127_2025_QH15_686325: Phạm vi điều chỉnh
     - 133_2025_QH15_675211: Phạm vi điều chỉnh
     - 135_2025_QH15_675213: Phạm vi điều chỉnh
     - 143_2025_QH15_681550: Phạm vi điều chỉnh
     - 148_2025_QH15_675262: Phạm vi điều chỉnh
     - sample v1: Đối Tượng Của Hợp Đồng
     - sample v2: Đối Tượng Của Hợp Đồng


In [25]:
# === Cell 10: Demo 4 — Hybrid Search (semantic + filter) ===
# ✏️ Thay đổi:
HYBRID_QUERY  = "quyền hạn và trách nhiệm của cơ quan nhà nước"
HYBRID_DOC_ID = "112_2025_QH15_586814"
HYBRID_TOP_K  = 3

results = search(
    HYBRID_QUERY, CHROMA_DIR, embedder=embedder,
    top_k=HYBRID_TOP_K,
    where={"doc_id": HYBRID_DOC_ID} if HYBRID_DOC_ID else None,
)

print(f'🔍 HYBRID: "{HYBRID_QUERY}"')
print(f"   Giới hạn: {HYBRID_DOC_ID or '(tất cả)'}  →  Top {HYBRID_TOP_K}\n")
for i, r in enumerate(results, 1):
    show_result(i, r)
print("─" * 65)

🔍 HYBRID: "quyền hạn và trách nhiệm của cơ quan nhà nước"
   Giới hạn: 112_2025_QH15_586814  →  Top 3

-----------------------------------------------------------------
  [1] Dieu 12 - Quản lý nhà nước về quy hoạch
       Score :  |  Score: -47.4415
       Nguon : 112_2025_QH15_586814  |  Chuong I
         Điều 12. Quản lý nhà nước về quy hoạch
         1. Chính phủ thống nhất quản lý nhà nước về quy hoạch trong phạm vi cả nước.
         2. Bộ Tài chính là cơ quan đầu mối giúp Chính phủ thực hiện quản lý nhà nước về quy hoạch.
         3. Các Bộ, Ủy ban nhân dân các cấp thực hiện quản lý nhà nước về quy hoạch trong phạm vi nhiệ
         ... [326 ky tu]
-----------------------------------------------------------------
  [2] Dieu 17 - Thẩm quyền tổ chức lập quy hoạch
       Score :  |  Score: -54.4834
       Nguon : 112_2025_QH15_586814  |  Chuong II
         Điều 17. Thẩm quyền tổ chức lập quy hoạch
         1. Chính phủ tổ chức lập quy hoạch tổng thể quốc gia, quy hoạch không gian biển